# PM-RL1 人评验收前置核查（2026-10-14）

## tl;dr
已核对40对原题、27个前缀、15个owner和17对表单间A/B换序。尚未收到实际Chen/Du导出，**人工有效数未知**；12题问题不能由面板计数推断。


## Context & Methods
只读冻结题包、映射和开发评分汇总，不解密任何答案，不运行模型。

### Key Assumptions
Chen/Du来源关系使用用户声明，实际文件未核实；不假定Chen等于rater_1。封存24对必须等最终测量冻结；当前只做原材料和分组核验。无答案数据，不计算置信区间，也不把40对当40个独立owner。

## Data
下面路径可显式修改，但输入必须是对应冻结原题包。本地私钥只由既有冻结哈希核对，不输出、复制或解密。


In [1]:
from pathlib import Path
import importlib.util, json
repo = Path('/home/tokkio/Metacom5_3').resolve()
audit_root = repo / 'project/outputs/pm_rl1/human_acceptance_20261014_v1'
packet_zip = Path('/home/tokkio/PM_RL1_40对人工校准包_20261001.zip')
spec = importlib.util.spec_from_file_location('packet_preflight', audit_root / 'audit_original_packet.py')
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
summary, rows, inputs = module.inspect(repo, packet_zip)
print(json.dumps({k:summary[k] for k in ['status','repository_commit','active_pairs','active_prefixes','active_owners','pair_groups','reversed_AB_between_forms','human_exports_received','human_preference_valid_count']}, ensure_ascii=False, indent=2))


{
  "status": "PREFLIGHT_PASS_HUMAN_EXPORT_REQUIRED",
  "repository_commit": "0c85538ad9a5f6065a7bc31ee0769cb76eaa0f28",
  "active_pairs": 40,
  "active_prefixes": 27,
  "active_owners": 15,
  "pair_groups": {
    "development": 16,
    "validation": 24
  },
  "reversed_AB_between_forms": 17,
  "human_exports_received": 0,
  "human_preference_valid_count": null
}


## Results
四个分层均是原题数量，不是验收通过的人评数量。


In [2]:
from collections import Counter
assert len(rows) == len({r['pair_id'] for r in rows}) == 40
assert len({r['rater_1_item_id'] for r in rows}) == len({r['rater_2_item_id'] for r in rows}) == 40
multiplicity = Counter(Counter(r['prefix_identity'] for r in rows).values())
assert multiplicity == {1:14, 2:13}
assert sum(r['rater_1_A_equals_rater_2']=='B' for r in rows) == 17
assert {r['group'] for r in rows if r['existing_first_scores_available']} == {'development'}
assert summary['human_preference_valid_count'] is None
assert summary['human_risk_valid_count'] is None
assert summary['human_problem_12_count'] is None
assert summary['human_answers_decrypted'] == summary['validation_answers_decrypted'] == 0
print(json.dumps({'strata':summary['strata'], 'prefix_multiplicity':dict(multiplicity), 'unverified_dashboard_counts':{'inputs':40,'judgments_recorded':29,'complete':28,'needs_review':12}, 'dashboard_is_not_human_answer_evidence':True}, ensure_ascii=False, indent=2))


{
  "strata": [
    {
      "group": "development",
      "kind": "natural",
      "pairs": 8,
      "owners": 3,
      "prefixes": 8,
      "first_scores_available_pairs": 8,
      "human_pairs_inspected": 0
    },
    {
      "group": "development",
      "kind": "control",
      "pairs": 8,
      "owners": 3,
      "prefixes": 8,
      "first_scores_available_pairs": 8,
      "human_pairs_inspected": 0
    },
    {
      "group": "validation",
      "kind": "natural",
      "pairs": 16,
      "owners": 12,
      "prefixes": 16,
      "first_scores_available_pairs": 0,
      "human_pairs_inspected": 0
    },
    {
      "group": "validation",
      "kind": "control",
      "pairs": 8,
      "owners": 8,
      "prefixes": 8,
      "first_scores_available_pairs": 0,
      "human_pairs_inspected": 0
    }
  ],
  "prefix_multiplicity": {
    "2": 13,
    "1": 14
  },
  "unverified_dashboard_counts": {
    "inputs": 40,
    "judgments_recorded": 29,
    "complete": 28,
    "needs_review":

## Takeaways
- 先取得现有导出包，再只开开发16对并逐字段验收；不能先重评或丢弃12题。
- Du派生记录不能增加样本量或证明独立一致性。
- 尚不能评价人机吻合或宣告奖励合格；最终24对仍封存。

本环境无nbformat/nbclient/nbconvert；代码单元已用标准Python在干净命名空间依序执行，并保存输出。只做了结构和数值校验，未做Jupyter前端视觉预览。需要原生Jupyter复核时，可在已有Jupyter环境运行：`jupyter nbconvert --execute --to notebook --output preflight_reexecuted.ipynb preflight_analysis.ipynb`。
